In [2]:

import re

# --------------------------------------------------
# 1. Parsing and AST Structures
# --------------------------------------------------

class Expr:
    def __init__(self, op, args=None):
        self.op = op
        self.args = args or []

    def __repr__(self):
        if not self.args:
            return self.op

        if self.op in ('&', '|', '->', '<->'):
            return f"({self.args[0]} {self.op} {self.args[1]})"

        if self.op == '~':
            return f"~{self.args[0]}"

        if self.op in ('forall', 'exists'):
            return f"{self.op} {self.args[0]}.{self.args[1]}"

        return f"{self.op}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return isinstance(other, Expr) and repr(self) == repr(other)

    def __hash__(self):
        return hash(repr(self))


def tokenize(s):
    return re.findall(
        r'forall|exists|<->|->|[().,~&|]|[A-Za-z0-9_]+',
        s
    )


def parse_fol(s):
    tokens = tokenize(s)

    def parse_iff():
        lhs = parse_imp()
        while tokens and tokens[0] == '<->':
            tokens.pop(0)
            lhs = Expr('<->', [lhs, parse_imp()])
        return lhs

    def parse_imp():
        lhs = parse_or()
        while tokens and tokens[0] == '->':
            tokens.pop(0)
            lhs = Expr('->', [lhs, parse_or()])
        return lhs

    def parse_or():
        lhs = parse_and()
        while tokens and tokens[0] == '|':
            tokens.pop(0)
            lhs = Expr('|', [lhs, parse_and()])
        return lhs

    def parse_and():
        lhs = parse_unary()
        while tokens and tokens[0] == '&':
            tokens.pop(0)
            lhs = Expr('&', [lhs, parse_unary()])
        return lhs

    def parse_unary():
        if not tokens:
            raise ValueError("Unexpected end of expression")

        if tokens[0] == '~':
            tokens.pop(0)
            return Expr('~', [parse_unary()])

        if tokens[0] in ('forall', 'exists'):
            q = tokens.pop(0)
            var = Expr(tokens.pop(0))

            if tokens and tokens[0] == '.':
                tokens.pop(0)

            return Expr(q, [var, parse_unary()])

        if tokens[0] == '(':
            tokens.pop(0)
            e = parse_iff()

            if not tokens or tokens.pop(0) != ')':
                raise ValueError("Missing closing parenthesis")

            return e

        name = tokens.pop(0)

        if tokens and tokens[0] == '(':
            tokens.pop(0)
            args = []

            if tokens and tokens[0] != ')':
                while True:
                    args.append(parse_unary())

                    if tokens and tokens[0] == ',':
                        tokens.pop(0)
                    else:
                        break

            if not tokens or tokens.pop(0) != ')':
                raise ValueError("Missing closing parenthesis")

            return Expr(name, args)

        return Expr(name)

    result = parse_iff()

    if tokens:
        raise ValueError("Unexpected tokens: " + str(tokens))

    return result


# --------------------------------------------------
# 2. Six Steps of CNF Conversion
# --------------------------------------------------

# Step 1: Eliminate implications

def step1_eliminate_implications(e):

    if e.op == '<->':
        a = step1_eliminate_implications(e.args[0])
        b = step1_eliminate_implications(e.args[1])

        return Expr('&', [
            Expr('|', [Expr('~', [a]), b]),
            Expr('|', [Expr('~', [b]), a])
        ])

    if e.op == '->':
        a = step1_eliminate_implications(e.args[0])
        b = step1_eliminate_implications(e.args[1])

        return Expr('|', [Expr('~', [a]), b])

    return Expr(
        e.op,
        [step1_eliminate_implications(a) for a in e.args]
    )


# Step 2: Move negation inward

def step2_move_negation_inward(e):

    if e.op == '~':
        sub = e.args[0]

        if sub.op == '~':
            return step2_move_negation_inward(sub.args[0])

        if sub.op == '&':
            return Expr('|', [
                step2_move_negation_inward(Expr('~', [a]))
                for a in sub.args
            ])

        if sub.op == '|':
            return Expr('&', [
                step2_move_negation_inward(Expr('~', [a]))
                for a in sub.args
            ])

        if sub.op == 'forall':
            return Expr('exists', [
                sub.args[0],
                step2_move_negation_inward(Expr('~', [sub.args[1]]))
            ])

        if sub.op == 'exists':
            return Expr('forall', [
                sub.args[0],
                step2_move_negation_inward(Expr('~', [sub.args[1]]))
            ])

        return Expr('~', [step2_move_negation_inward(sub)])

    return Expr(
        e.op,
        [step2_move_negation_inward(a) for a in e.args]
    )


# Step 3: Standardize variables

var_counter = 0


def step3_standardize_variables(e, env=None):
    global var_counter

    env = dict(env or {})

    if e.op in ('forall', 'exists'):
        var_counter += 1

        new_var = Expr(f"{e.args[0].op}_{var_counter}")
        env[e.args[0].op] = new_var

        return Expr(e.op, [
            new_var,
            step3_standardize_variables(e.args[1], env)
        ])

    if not e.args and e.op in env:
        return env[e.op]

    return Expr(
        e.op,
        [step3_standardize_variables(a, env) for a in e.args]
    )


# Step 4: Skolemization

skolem_counter = 0


def step4_skolemize(e, univ_vars=None):
    global skolem_counter

    univ_vars = list(univ_vars or [])

    if e.op == 'forall':
        return Expr('forall', [
            e.args[0],
            step4_skolemize(e.args[1], univ_vars + [e.args[0]])
        ])

    if e.op == 'exists':
        skolem_counter += 1

        if univ_vars:
            skolem_term = Expr(f"Sk{skolem_counter}", univ_vars)
        else:
            skolem_term = Expr(f"SkConst{skolem_counter}")

        return step4_skolemize(
            substitute(e.args[1], {e.args[0].op: skolem_term}),
            univ_vars
        )

    return Expr(
        e.op,
        [step4_skolemize(a, univ_vars) for a in e.args]
    )


# Step 5: Drop universal quantifiers

def step5_drop_universal(e):

    if e.op == 'forall':
        return step5_drop_universal(e.args[1])

    return Expr(
        e.op,
        [step5_drop_universal(a) for a in e.args]
    )


# Step 6: Distribute OR over AND

def step6_distribute_and_cnf(e):

    e = Expr(
        e.op,
        [step6_distribute_and_cnf(a) for a in e.args]
    )

    if e.op == '|':
        a, b = e.args

        if a.op == '&':
            return step6_distribute_and_cnf(
                Expr('&', [
                    Expr('|', [a.args[0], b]),
                    Expr('|', [a.args[1], b])
                ])
            )

        if b.op == '&':
            return step6_distribute_and_cnf(
                Expr('&', [
                    Expr('|', [a, b.args[0]]),
                    Expr('|', [a, b.args[1]])
                ])
            )

    return e


# Convert CNF expression into clauses

def to_clauses(e):

    if e.op == '&':
        return to_clauses(e.args[0]) + to_clauses(e.args[1])

    def get_disjuncts(d):
        if d.op == '|':
            return get_disjuncts(d.args[0]) + get_disjuncts(d.args[1])

        return [d]

    return [set(get_disjuncts(e))]


# Complete FOL to CNF conversion

def fol_to_cnf(formula_str):

    global var_counter, skolem_counter

    e = parse_fol(formula_str)

    e = step1_eliminate_implications(e)
    e = step2_move_negation_inward(e)
    e = step3_standardize_variables(e)
    e = step4_skolemize(e)
    e = step5_drop_universal(e)
    e = step6_distribute_and_cnf(e)

    return to_clauses(e)


# --------------------------------------------------
# 3. Unification and Resolution Engine
# --------------------------------------------------

# Check whether an expression is a variable

def is_var(x):
    return (
        isinstance(x, Expr)
        and not x.args
        and x.op[0].islower()
    )


# Substitute variables

def substitute(e, theta):

    if is_var(e) and e.op in theta:
        return substitute(theta[e.op], theta)

    return Expr(
        e.op,
        [substitute(a, theta) for a in e.args]
    )


# Occurs check

def occurs_check(v, x, theta):

    if v == x:
        return True

    if is_var(x) and x.op in theta:
        return occurs_check(v, theta[x.op], theta)

    return any(
        occurs_check(v, a, theta)
        for a in x.args
    )


# Unification algorithm

def unify(x, y, theta=None):

    if theta is None:
        theta = {}

    if x == y:
        return theta

    if is_var(x):

        if x.op in theta:
            return unify(theta[x.op], y, theta)

        if occurs_check(x, y, theta):
            return None

        return {**theta, x.op: y}

    if is_var(y):

        if y.op in theta:
            return unify(x, theta[y.op], theta)

        if occurs_check(y, x, theta):
            return None

        return {**theta, y.op: x}

    if x.op != y.op or len(x.args) != len(y.args):
        return None

    for a, b in zip(x.args, y.args):
        theta = unify(a, b, theta)

        if theta is None:
            return None

    return theta


# Rename variables in a clause to avoid variable clashes

def standardize_clause(clause, suffix):

    mapping = {}

    def rename(e):

        if is_var(e):
            if e.op not in mapping:
                mapping[e.op] = Expr(f"{e.op}_{suffix}")

            return mapping[e.op]

        return Expr(e.op, [rename(a) for a in e.args])

    return {rename(literal) for literal in clause}


# Resolve two clauses

def resolve_pair(c1, c2):

    resolvents = []

    for l1 in c1:
        for l2 in c2:

            theta = None

            if l1.op == '~' and l2.op != '~':
                theta = unify(l1.args[0], l2, {})

            elif l2.op == '~' and l1.op != '~':
                theta = unify(l1, l2.args[0], {})

            if theta is not None:

                new_c = {
                    substitute(l, theta)
                    for l in (c1 - {l1}) | (c2 - {l2})
                }

                resolvents.append(frozenset(new_c))

    return resolvents


# FOL Resolution algorithm

def fol_resolution(kb_list, query_str, max_steps=100):

    clauses = set()

    # Convert all KB premises into CNF clauses

    for stmt in kb_list:
        for c in fol_to_cnf(stmt):
            clauses.add(frozenset(c))

    # Negate the query and add it to KB

    for c in fol_to_cnf(f"~({query_str})"):
        clauses.add(frozenset(c))

    # Resolution process

    for step in range(max_steps):

        clause_list = list(clauses)
        new = set()

        for i in range(len(clause_list)):

            for j in range(i + 1, len(clause_list)):

                # Standardize clauses apart
                c1 = standardize_clause(clause_list[i], f"{step}_{i}")
                c2 = standardize_clause(clause_list[j], f"{step}_{j}")

                for res in resolve_pair(c1, c2):

                    # Empty clause derived
                    if len(res) == 0:
                        return True

                    if res not in clauses:
                        new.add(res)

        if not new:
            return False

        clauses |= new

    return False


# --------------------------------------------------
# 4. User Input and Output
# --------------------------------------------------

if __name__ == "__main__":

    print("Syntax: forall x. P(x), exists x. P(x), ~ (NOT), & (AND), | (OR), -> (IMP), <-> (IFF)")

    print("Rule: Variables start with lowercase (x, y); Constants/Predicates start with uppercase (John, Human).")

    n = int(input("Number of KB premises: "))

    kb = [
        input(f"Premise {i+1}: ")
        for i in range(n)
    ]

    query = input("Query to prove: ")

    can_prove = fol_resolution(kb, query)

    print("\n--- RESULT ---")

    print(
        f"Resolution Tree Formed: "
        f"{'Yes (Empty clause [] derived)' if can_prove else 'No'}"
    )

    print(f"Query Proven: {'TRUE' if can_prove else 'FALSE'}")


Syntax: forall x. P(x), exists x. P(x), ~ (NOT), & (AND), | (OR), -> (IMP), <-> (IFF)
Rule: Variables start with lowercase (x, y); Constants/Predicates start with uppercase (John, Human).
Number of KB premises: 7
Premise 1: forall x. (Food(x) -> Likes(John, x))
Premise 2: Food(Apple) & Food(Vegetables)
Premise 3: forall x. forall y. ((Eats(x, y) & ~Killed(x)) -> Food(y))
Premise 4: Eats(Anil, Peanuts) & Alive(Anil)
Premise 5: forall x. (Eats(Anil, x) -> Eats(Harry, x))
Premise 6: forall x. (Alive(x) -> ~Killed(x))
Premise 7: forall x. (~Killed(x) -> Alive(x))
Query to prove: Likes(John, Peanuts)

--- RESULT ---
Resolution Tree Formed: Yes (Empty clause [] derived)
Query Proven: TRUE
